# FEC Simulation — EEP vs UEP
**Bernoulli packet loss at 1%, 5%, 10%, 20%**

UEP guarantees minimum 1 redundant copy per stream, surplus goes to vocals.

Quality metric: SNR scaled 0-100 (pure numpy).

**Before running:** upload `separated_dsp_nn.zip` when prompted.

In [ ]:
!pip install soundfile librosa -q
print('Dependencies installed')


In [ ]:
from google.colab import files
import zipfile, os, shutil
print('Upload separated_dsp_nn.zip:')
uploaded = files.upload()
for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('.')
        print(f'Extracted: {fname}')
os.makedirs('separated_dsp_nn', exist_ok=True)
for d in os.listdir('.'):
    if os.path.isdir(d) and d not in ['separated_dsp_nn','fec_output','sample_data']:
        dest = f'separated_dsp_nn/{d}'
        if not os.path.exists(dest):
            shutil.move(d, dest)
            print(f'Moved {d} -> {dest}')
print('Contents:', os.listdir('separated_dsp_nn'))


In [ ]:
"""
FEC Simulation — EEP vs UEP with ViSQOL Quality Metric
=======================================================
Loads separated vocal and instrumental streams from separated_dsp_nn/
Simulates Bernoulli packet loss at 1%, 5%, 10%, 20% loss rates.

EEP (Equal Error Protection):
  Both streams receive the same number of redundant copies — naive baseline.

UEP (Unequal Error Protection):
  Redundancy allocated dynamically per 20ms chunk based on vocal confidence
  score (energy in 300-3000 Hz). Vocals get more protection; instrumentals
  accept higher loss in exchange for better vocal recovery.

FEC model: repetition coding.
  Each packet gets R redundant copies. Recovery if any copy survives.
  P(total loss) = loss_rate ^ (R + 1)

ViSQOL: compares recovered audio vs original clean separation (0-100).
  Requires 48kHz — audio is resampled before scoring.
  Falls back to PESQ if visqol-python is unavailable.

Output: fec_output/<song>/loss_<X>pct_<EEP|UEP>/vocals.wav + instrumentals.wav
"""

import os
import numpy as np
import soundfile as sf
import librosa
from pathlib import Path

SR            = 44100
VISQOL_SR     = 48000
CHUNK_MS      = 20
CHUNK_LEN     = int(SR * CHUNK_MS / 1000)
LOSS_RATES    = [0.01, 0.05, 0.10, 0.20]
REDUND_BUDGET = 3    # total extra copies per stream pair
MIN_REDUND    = 1    # guaranteed minimum for each stream
VOCAL_BIAS    = 0.10
N_FFT         = 2048
HOP_STFT      = 512


# ── ViSQOL setup ──────────────────────────────────────────────────────────────

def load_visqol():
    try:
        import visqol
        print("  ViSQOL loaded successfully")
        return visqol
    except ImportError:
        print("  visqol-python not found — falling back to PESQ")
        return None


def mos_to_100(mos: float) -> float:
    return round((mos - 1.0) / 4.0 * 100, 1)


def compute_visqol(ref: np.ndarray, deg: np.ndarray, visqol_lib) -> float:
    if visqol_lib is None:
        return None
    try:
        from visqol import visqol_lib_py
        from visqol.pb2 import visqol_config_pb2

        ref_48 = librosa.resample(ref, orig_sr=SR, target_sr=VISQOL_SR)
        deg_48 = librosa.resample(deg, orig_sr=SR, target_sr=VISQOL_SR)

        config = visqol_config_pb2.VisqolConfig()
        config.audio.sample_rate = VISQOL_SR
        config.options.use_speech_scoring = False

        api = visqol_lib_py.VisqolApi()
        api.Create(config)

        result = api.Measure(ref_48.astype(np.float64), deg_48.astype(np.float64))
        return mos_to_100(result.moslqo)
    except Exception:
        return None


def compute_snr_score(ref: np.ndarray, deg: np.ndarray) -> float:
    """
    Pure numpy SNR scaled to 0-100. No C extensions — cannot crash kernel.
    SNR = 10*log10(signal power / noise power), clamped to [0, 40] dB → [0, 100].
    Perfect recovery (no loss) = 100. Complete loss (zeros) ≈ 0.
    """
    n   = min(len(ref), len(deg))
    ref = ref[:n]
    deg = deg[:n]
    noise      = ref - deg
    sig_power  = np.mean(ref ** 2) + 1e-10
    noise_power= np.mean(noise ** 2) + 1e-10
    snr_db     = 10 * np.log10(sig_power / noise_power)
    return round(float(np.clip(snr_db / 40.0 * 100, 0, 100)), 1)


def quality_score(ref: np.ndarray, deg: np.ndarray, visqol_lib) -> str:
    score = compute_visqol(ref, deg, visqol_lib)
    if score is None:
        score = compute_snr_score(ref, deg)
    return f"{score:>5.1f}" if score is not None else "  N/A"


# ── audio I/O ─────────────────────────────────────────────────────────────────

def load_wav(path: Path) -> np.ndarray:
    audio, _ = sf.read(str(path))
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    return audio.astype(np.float32)


def write_wav(path: Path, audio: np.ndarray, sr: int = SR):
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak
    sf.write(str(path), (audio * 32767).astype(np.int16), sr)


def chunk_audio(audio: np.ndarray, chunk_len: int):
    n        = len(audio)
    n_chunks = int(np.ceil(n / chunk_len))
    padded   = np.zeros(n_chunks * chunk_len, dtype=np.float32)
    padded[:n] = audio
    return padded.reshape(n_chunks, chunk_len), n


# ── per-chunk vocal confidence ─────────────────────────────────────────────────

def vocal_confidence_score(chunk: np.ndarray) -> float:
    S      = np.abs(librosa.stft(chunk, n_fft=N_FFT, hop_length=HOP_STFT))
    freqs  = librosa.fft_frequencies(sr=SR, n_fft=N_FFT)
    v_bins = (freqs >= 300) & (freqs <= 3000)
    return float(S[v_bins].sum() / (S.sum() + 1e-8))


# ── FEC simulation ─────────────────────────────────────────────────────────────

def all_copies_lost(n_copies: int, loss_rate: float, rng) -> bool:
    return bool((rng.random(n_copies) < loss_rate).all())


def simulate_fec(vocal_chunks, inst_chunks, confidence, loss_rate, rng, mode):
    n         = len(vocal_chunks)
    vocal_out = np.zeros_like(vocal_chunks)
    inst_out  = np.zeros_like(inst_chunks)
    v_rec = i_rec = 0

    for idx in range(n):
        if mode == "eep":
            v_r = REDUND_BUDGET // 2
            i_r = REDUND_BUDGET - v_r
        else:  # uep — guaranteed minimum + surplus allocated by confidence
            surplus = REDUND_BUDGET - 2 * MIN_REDUND
            v_score = confidence[idx] + VOCAL_BIAS
            i_score = max(1.0 - confidence[idx], 0.0)
            total   = v_score + i_score
            v_r     = MIN_REDUND + round((v_score / total) * surplus)
            i_r     = REDUND_BUDGET - v_r

        if not all_copies_lost(v_r + 1, loss_rate, rng):
            vocal_out[idx] = vocal_chunks[idx]
            v_rec += 1

        if not all_copies_lost(i_r + 1, loss_rate, rng):
            inst_out[idx] = inst_chunks[idx]
            i_rec += 1

    return vocal_out, inst_out, v_rec / n, i_rec / n


# ── per-song runner ────────────────────────────────────────────────────────────

def run_song(song_dir: Path, output_base: Path, visqol_lib):
    vocal_path = song_dir / "streaming_vocals.wav"
    inst_path  = song_dir / "streaming_instrumentals.wav"

    if not vocal_path.exists() or not inst_path.exists():
        print(f"  Skipping {song_dir.name} — missing separated files")
        return

    print(f"\n{'='*72}")
    print(f"  {song_dir.name}")
    print(f"{'='*72}")

    vocals = load_wav(vocal_path)
    instrs = load_wav(inst_path)
    n      = min(len(vocals), len(instrs))
    vocals, instrs = vocals[:n], instrs[:n]

    vocal_chunks, orig_len = chunk_audio(vocals, CHUNK_LEN)
    inst_chunks,  _        = chunk_audio(instrs, CHUNK_LEN)
    n_chunks = len(vocal_chunks)
    print(f"  Packets: {n_chunks} × {CHUNK_MS}ms")

    print("  Computing per-chunk vocal confidence scores...")
    confidence = np.array([vocal_confidence_score(c) for c in vocal_chunks])
    print(f"  Mean confidence: {confidence.mean():.3f}")

    print(f"\n  {'Loss':>6}  {'Mode':<6}  "
          f"{'Vocal rec':>10}  {'Inst rec':>9}  "
          f"{'V.Quality/100':>14}  {'I.Quality/100':>14}")
    print(f"  {'─'*66}")

    rng = np.random.default_rng(42)

    for loss_rate in LOSS_RATES:
        for mode in ["eep", "uep"]:
            v_out, i_out, v_rec, i_rec = simulate_fec(
                vocal_chunks, inst_chunks, confidence, loss_rate, rng, mode
            )
            v_flat = v_out.flatten()[:orig_len]
            i_flat = i_out.flatten()[:orig_len]

            v_q = quality_score(vocals, v_flat, visqol_lib)
            i_q = quality_score(instrs, i_flat, visqol_lib)

            print(f"  {loss_rate*100:>5.0f}%  {mode.upper():<6}  "
                  f"{v_rec*100:>9.1f}%  {i_rec*100:>8.1f}%  "
                  f"{v_q:>14}  {i_q:>14}")

            out_dir = output_base / song_dir.name / f"loss_{int(loss_rate*100):02d}pct_{mode.upper()}"
            os.makedirs(out_dir, exist_ok=True)
            write_wav(out_dir / "vocals.wav",        v_flat)
            write_wav(out_dir / "instrumentals.wav", i_flat)

        print()


# ── main ──────────────────────────────────────────────────────────────────────

def main():
    input_base  = Path("separated_dsp_nn")
    output_base = Path("fec_output")

    if not input_base.exists():
        print(f"'{input_base}/' not found.")
        print("Run notebook_dsp_nn.ipynb first and unzip separated_dsp_nn.zip here.")
        return

    songs = sorted(d for d in input_base.iterdir() if d.is_dir())
    if not songs:
        print(f"No song folders found in {input_base}/")
        return

    print("=" * 72)
    print("  FEC SIMULATION — EEP vs UEP with ViSQOL Quality Metric")
    print(f"  Model:            Bernoulli independent packet loss")
    print(f"  Loss rates:       {[f'{r*100:.0f}%' for r in LOSS_RATES]}")
    print(f"  Redundancy budget:{REDUND_BUDGET} extra copies per stream pair")
    print(f"  Vocal bias:       +{VOCAL_BIAS} at equal confidence")
    print(f"  Quality metric:   ViSQOL 0-100 vs original clean separation")
    print("=" * 72)

    visqol_lib = load_visqol()

    for song_dir in songs:
        run_song(song_dir, output_base, visqol_lib)

    print("=" * 72)
    print(f"  All done. Recovered audio in fec_output/")
    print("=" * 72)


if __name__ == "__main__":
    main()


In [ ]:
import sys, warnings
sys.modules['visqol'] = None
sys.modules['pesq'] = None
warnings.filterwarnings('ignore')
main()


In [ ]:
import shutil
from google.colab import files
shutil.make_archive('fec_output', 'zip', 'fec_output')
files.download('fec_output.zip')
